In [90]:
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec  #grid pels subplots
from matplotlib import colors         #colors
from matplotlib.colors import LogNorm   #normalitza a 0-1 en escala logarítmica 
from matplotlib import patches     #figures
from mpl_toolkits.mplot3d import Axes3D   #eixos en 3D
from mpl_toolkits.axes_grid1 import make_axes_locatable  #per canviar la posició dels eixos
from matplotlib.ticker import NullFormatter   #marques(tics) sense etiquetes en els eixos

from astroML.density_estimation import XDGMM
from astroML.plotting.tools import draw_ellipse
from astroML.plotting import scatter_contour
from astroML.crossmatch import crossmatch
from astroML.datasets import fetch_sdss_S82standards, fetch_imaging_sample
from astroML.stats import sigmaG
from astroML.utils.decorators import pickle_results
import os
import pickle

import astropy.table     #paquet per manejar taules de dades
from astropy.table import Table, Column, MaskedColumn   #importa taules, columnes i columnes que emmascaren dades invàlides
from astropy.visualization import astropy_mpl_style  #visualització 
from scipy.stats import gaussian_kde  #representation of a kernel-density estimate using Gaussian kernels.
import seaborn as sns  #llibreria per fer gràfics estadístics
import os.path   #per implementar diferents funcions amb pathnames ("dreceres")

from time import time   #mòdul de funcions de time access 
from sklearn import manifold, datasets #manifold: algoritme de dimensionality reduction 
                                       #sklearn (sci-kit learn): llibreria de Python per machine learning

import obtain_data   #per importar dades d'altres fitxers


import importlib   #package per importar coses a python
importlib.reload(obtain_data)

galah_rc = obtain_data.galah_dr3_rc()  

galah_rc.get_ndimspace(feh=True, norm="stdev")

X = galah_rc.X            #(10941, 24)
Xerr = galah_rc.Xerr1         #(10941, 24)
Xcov = np.zeros(Xerr.shape + Xerr.shape[-1:])
Xcov[:, range(Xerr.shape[1]), range(Xerr.shape[1])] = Xerr ** 2    #(10941, 24, 24)

In [91]:
def show_abundances(n_components):
    N = 0
    fe_h_mean = 0
    Z = [26,8,11,12,13,14,19,20,21,22,23,24,25,27,28,29,30,39,40,56,57,58,60,63]
    Z_labels = ["Fe","O","Na","Mg","Al","Si","K","Ca","Sc","Ti","V","Cr","Mn","Co","Ni","Cu","Zn","Y","Zr","Ba","La","Ce","Nd","Eu"]
    abundance_titles   = ['[Fe/H]' ,  '[O/Fe]',   '[Na/Fe]', '[Mg/Fe]',
                          '[Al/Fe]',  '[Si/Fe]',  '[K/Fe]',  '[Ca/Fe]',
                          '[Sc/Fe]',  '[Ti/Fe]',  '[V/Fe]' , '[Cr/Fe]',
                          '[Mn/Fe]',  '[Co/Fe]',  '[Ni/Fe]', '[Cu/Fe]',
                          '[Zn/Fe]',  '[Y/Fe]',  ' [Zr/Fe]', '[Ba/Fe]', 
                          '[La/Fe]',  '[Ce/Fe]',  '[Nd/Fe]', '[Eu/Fe]']
    ck = ["k", "r", "gold", "g", "b",
          "orange", "cyan", "lime", "m", "yellow",
          "indianred", "hotpink", "peru", "cornflowerblue", "olivedrab",
          "grey", "turquoise", "lightpink", "navy", "khaki",
          "darkgreen", "crimson", "deepskyblue", "sandybrown", "limegreen",
          "deeppink", "dodgerblue", "rebeccapurple", "teal", "magenta"]
    print('  \t     N_stars   \t    Metallicity     \t    max. [X]       \t     min. [X]')
    for ii in range (n_components):  # pels 25 clusters
        n = collections.Counter(maxprob_per_star)[ii] # n stars per cluster
        X_mean = np.sum(X[maxprob_per_star==ii], axis=0) / n  # mean abundances per cluster
        fe_h = X_mean[0]
        x_max = abundance_titles[np.argmax(X_mean)]
        x_min = abundance_titles[np.argmin(X_mean)]
        
        N += n
        fe_h_mean += fe_h / n_components

        print('cluster_'+str(ii)+' ('+ck[ii]+'): ', n, 'stars,\t','[Fe/H] =',"{:.3f}".format(fe_h),'\t',x_max,'=',"{:.3f}".format(X_mean[np.argmax(X_mean)]),'\t', x_min,'=', "{:.3f}".format(X_mean[np.argmin(X_mean)]) )    
    print('Total sample: ', N, 'stars,\t','[Fe/H] =',"{:.3f}".format(fe_h_mean)) 
    print('\n')
    print('\n')

In [92]:
for n_components in [5,10,20,25]:

    @pickle_results('XD_'+str(n_components)+'clusters.pkl')
    def compute_XD(n_components, rseed=0, max_iter=100, verbose=True):
        np.random.seed(rseed)
        clf = XDGMM(n_components, max_iter=max_iter, tol=1E-5, verbose=verbose)
        clf.fit(X, Xcov) 
        return clf
    # Fit the model on training set
    clf = compute_XD(n_components)

    # Probabilities for each star to belong to each group
    logprob = (clf.logprob_a(X, Xcov))
    prob_per_star = np.exp(logprob) / np.sum(np.exp(logprob), axis=1)[:, np.newaxis]  #(10941 estrelles, 25 probs)
    maxprob_per_star = np.argmax(prob_per_star, axis=1)   #suma en horitzontal

    show_abundances(n_components)

@pickle_results: using precomputed results from 'XD_5clusters.pkl'
  	     N_stars   	    Metallicity     	    max. [X]       	     min. [X]
cluster_0 (k):  1273 stars,	 [Fe/H] = -0.024 	 [V/Fe] = 0.445 	 [Ce/Fe] = -0.081
cluster_1 (r):  396 stars,	 [Fe/H] = -0.309 	 [Ba/Fe] = 0.651 	 [Fe/H] = -0.309
cluster_2 (gold):  4057 stars,	 [Fe/H] = -0.037 	 [V/Fe] = 0.394 	 [Y/Fe] = -0.094
cluster_3 (g):  2306 stars,	 [Fe/H] = -0.308 	 [O/Fe] = 0.375 	 [Fe/H] = -0.308
cluster_4 (b):  2909 stars,	 [Fe/H] = -0.213 	 [Ba/Fe] = 0.375 	 [Fe/H] = -0.213
Total sample:  10941 stars,	 [Fe/H] = -0.178




@pickle_results: using precomputed results from 'XD_10clusters.pkl'
  	     N_stars   	    Metallicity     	    max. [X]       	     min. [X]
cluster_0 (k):  1394 stars,	 [Fe/H] = -0.159 	 [V/Fe] = 0.368 	 [Fe/H] = -0.159
cluster_1 (r):  2172 stars,	 [Fe/H] = -0.073 	 [Ba/Fe] = 0.346 	 [Ce/Fe] = -0.085
cluster_2 (gold):  1777 stars,	 [Fe/H] = 0.013 	 [V/Fe] = 0.510 	 [Y/Fe] = -0.149
cluster_3 (g):  109